In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [16]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

C:\Users\johnb\AppData\Local\Temp\ipykernel_27296\2233616850.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [17]:
pip install -U langchain-tavily

Note: you may need to restart the kernel to use updated packages.


In [18]:
from getpass import getpass
import os

os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")

Enter Tavily API Key:  ········


In [19]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)

In [40]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [20]:
!pip install -U langgraph-checkpoint-sqlite

In [21]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [23]:
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [24]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [25]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")

Enter OpenAI API Key:  ········
Enter Tavily API Key:  ········


In [26]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [27]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [28]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [29]:
thread = {"configurable": {"thread_id": "1"}}

In [30]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='Could you please specify the date or time range for which you would like to know the weather in San Francisco? Are you asking for the current weather conditions or a forecast for a specific date or period?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 1259, 'total_tokens': 1300, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_727a239e01', 'id': 'chatcmpl-DkvFZogOZnnJerDjjzjm8AjOo9O4U', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e74c4-fecf-7f93-a44b-498c771afe89-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1259, 'output_tokens': 41, 'total_tokens': 1300, 'input_token_details': {'audio'

In [31]:
from getpass import getpass
import os

os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")

Enter Tavily API Key:  ········


In [32]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)

In [33]:
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 1314, 'total_tokens': 1338, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_727a239e01', 'id': 'chatcmpl-DkvGXxvCrUCpo9b07LbJMZ5JPchbf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e74c5-e830-7950-8fb9-0a17adc2c2ba-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather San Francisco', 'search_depth': 'fast'}, 'id': 'call_antve3oxmLWwFlIe0phbJN3J', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1314, 'output_tokens': 24, 'total_tokens': 1338, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_t

In [ ]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

In [52]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Could you please provide more context or specify what you are comparing to determine which is warmer? Are you comparing two specific places, materials, seasons, or something else?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 1257, 'total_tokens': 1291, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1152}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_727a239e01', 'id': 'chatcmpl-DkZ4jk9u4YWONgScTaXrjtzzrzadc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6fb0-5d91-7e22-987e-83a476d30b65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1257, 'output_tokens': 34, 'total_tokens': 1291, 'input_token_details': {'audio': 0, 'cach

In [22]:
!pip install -U langgraph-checkpoint-sqlite aiosqlite

In [23]:
!pip install -U langgraph-checkpoint-sqlite aiosqlite

In [24]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [25]:
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [26]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)

        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )

        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")

        self.graph = graph.compile(checkpointer=checkpointer)

        self.tools = {t.name: t for t in tools}

        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state["messages"]

        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        message = self.model.invoke(messages)

        return {"messages": [message]}

    def exists_action(self, state: AgentState):
        result = state["messages"][-1]

        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls

        results = []

        for t in tool_calls:

            print(f"Calling: {t}")

            if t["name"] not in self.tools:
                print("Bad tool name.")

                result = "Bad tool name, retry."

            else:
                result = self.tools[t["name"]].invoke(t["args"])

            results.append(
                ToolMessage(
                    tool_call_id=t["id"],
                    name=t["name"],
                    content=str(result)
                )
            )

        print("Back to the model!")

        return {"messages": results}

In [27]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [9]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")

Enter OpenAI API Key:  ········
Enter Tavily API Key:  ········


In [10]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o")

In [11]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)

In [12]:
prompt = """
You are a smart research assistant. Use the search engine to look up information.

You are allowed to make multiple calls (either together or in sequence).

Only look up information when you are sure of what you want.

If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

In [13]:
from langgraph.graph import StateGraph, END

In [14]:
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

NameError: name 'Agent' is not defined

In [15]:
from langchain_core.messages import HumanMessage

In [35]:
from langchain_core.messages import SystemMessage

In [36]:
from langchain_core.messages import ToolMessage

In [37]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}

async for event in abot.graph.astream_events(
    {"messages": messages},
    thread,
    version="v2"
):
    kind = event["event"]

    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content

        if content:
            print(content, end="")

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather San Francisco', 'search_depth': 'fast'}, 'id': 'call_UDpnGPsYyyR2WL1E9DV2Rl2k', 'type': 'tool_call'}
Back to the model!
The current weather in San Francisco is partly cloudy with a temperature of around 22° Celsius, dropping to about 12° Celsius later in the day. It is expected to remain dry throughout the evening.